In [0]:
#**Section 1: Load and inspect**

base = "/Volumes/cs483eviction/raw/cook_county_foreclosures"
path = f"{base}/cook_county_foreclosures_2012.json"

raw = (spark.read.option("multiLine", True)
            .json(path))

print("foreclosure rows:", raw.count())
raw.printSchema()
display(raw.limit(5))


#**Section 2: Clean and derive columns**


from pyspark.sql import functions as F

fc = (raw
      .withColumnRenamed("Recorded Date", "recorded_date")
      .withColumnRenamed("Execution Date", "execution_date")
      .withColumnRenamed("Doc Number", "doc_number")
      .withColumnRenamed("Doc Type", "doc_type")
      .withColumnRenamed("Consideration Amount", "consideration_amount")
      .withColumnRenamed("Location 1", "location_1")
      .withColumn("recorded_date", F.to_timestamp("recorded_date", "MM/dd/yyyy"))
      .withColumn("execution_date",
          F.when(F.year(F.to_timestamp("execution_date", "MM/dd/yyyy")).between(1900, 2100),
               F.to_timestamp("execution_date", "MM/dd/yyyy")))
      .withColumn("pin_clean", F.lpad(F.regexp_replace("Pin", "-", ""), 14, "0"))
      .withColumn("year", F.year("recorded_date"))
      .withColumn("recorded_month", F.date_format("recorded_date", "yyyy-MM"))
      .withColumn("month_start", F.trunc("recorded_date", "month"))
      .withColumn("latitude", F.nullif(F.regexp_extract("location_1", r"\(([^,]+),", 1), F.lit("")).cast("double"))
      .withColumn("longitude", F.nullif(F.regexp_extract("location_1", r",\s*(-?[0-9.]+)\)", 1), F.lit("")).cast("double"))
      .withColumn("has_location",
          F.col("latitude").isNotNull() & F.col("longitude").isNotNull())
      .withColumn("execution_missing", F.col("execution_date").isNull())
      .withColumn("consideration_missing", F.col("consideration_amount").isNull()))

print("rows after dedupe:", fc.count())
display(fc.select("pin_clean", "doc_number", "doc_type", "recorded_date",
                  "month_start", "has_location").limit(20))


#**Section 3: Missing-value check**


missing = fc.select([F.sum(F.col(c).isNull().cast("int")).alias(c)
                     for c in fc.columns])
display(missing)


#**Section 4: Summaries**


monthly = (fc.groupBy("year", "recorded_month", "month_start")
             .agg(F.count("*").alias("filings_total"),
                  F.countDistinct("pin_clean").alias("unique_pins"),
                  F.sum(F.col("has_location").cast("int")).alias("geocoded_filings"))
             .withColumn("geocoded_share", F.col("geocoded_filings") / F.col("filings_total"))
             .orderBy("recorded_month"))

by_type = (fc.groupBy("doc_type")
             .agg(F.count("*").alias("filings_total"))
             .orderBy(F.desc("filings_total")))

display(monthly)
display(by_type)


#**Section 5: Save to Delta table**


(fc.write.format("delta")
   .mode("overwrite")
   .option("overwriteSchema", "true")
   .saveAsTable("cs483eviction.default.cook_foreclosures_2012"))

print("saved rows:", spark.table("cs483eviction.default.cook_foreclosures_2012").count())


#**Section 6: Export to JSON**


df = spark.table("cs483eviction.default.cook_foreclosures_2012").toPandas()
df.to_json("/Volumes/cs483eviction/raw/lcbh_raw/cook_foreclosures_2012.json",
           orient="records", lines=True, date_format="iso")
print("exported", len(df), "rows")

# Note: Translated code to Spark using AI such as Microsoft Copilot and ChatGPT

In [0]:

# ==========================================
# SECTION 7: DATA QUALITY DIAGNOSTICS
# ==========================================

from pyspark.sql import functions as F

total_rows = fc.count()

original_columns = [
    "Pin",
    "doc_number",
    "doc_type",
    "recorded_date",
    "execution_date",
    "consideration_amount",
    "location_1"
]

missing_counts = fc.select([
    F.count(
        F.when(F.col(c).isNull(), 1)
    ).alias(c)
    for c in original_columns
])

print("MISSING VALUES")
display(missing_counts)

missing_percentages = fc.select([
    F.round(
        F.count(F.when(F.col(c).isNull(), 1))
        / F.lit(total_rows) * 100,
        2
    ).alias(c)
    for c in original_columns
])

print("MISSING VALUE PERCENTAGES")
display(missing_percentages)

exact_duplicates = total_rows - fc.dropDuplicates().count()

repeated_doc_numbers = (
    total_rows - fc.select("doc_number").distinct().count()
)

repeated_pins = (
    total_rows - fc.select("pin_clean").distinct().count()
)

repeated_doc_pin = (
    total_rows -
    fc.select("doc_number", "pin_clean").distinct().count()
)

print("DUPLICATE CHECKS")
print("Exact duplicate rows:", exact_duplicates)
print("Repeated document number occurrences:", repeated_doc_numbers)
print("Repeated PIN occurrences:", repeated_pins)
print("Repeated document/PIN combinations:", repeated_doc_pin)

date_range = fc.agg(
    F.min("recorded_date").alias("earliest_recorded_date"),
    F.max("recorded_date").alias("latest_recorded_date")
)

print("RECORDED DATE RANGE")
display(date_range)

outside_2012 = fc.filter(
    F.year("recorded_date") != 2012
).count()

print("Records outside 2012:", outside_2012)

execution_anomalies = fc.filter(
    F.col("execution_date") > F.col("recorded_date")
).count()

print("Execution dates later than recorded dates:", execution_anomalies)

print("DATA TYPES")
fc.printSchema()


In [0]:
# Section 8: Descriptive statistics

eda = (
    fc
    .withColumn(
        "consideration_amount_numeric",
        F.expr("""
            try_cast(
                nullif(
                    regexp_replace(
                        trim(cast(consideration_amount as string)),
                        '[$,]',
                        ''
                    ),
                    ''
                ) AS DOUBLE
            )
        """)
    )
    .withColumn(
        "latitude_eda",
        F.expr("""
            try_cast(
                regexp_extract(location_1, '\\\\((-?[0-9.]+),', 1)
                AS DOUBLE
            )
        """)
    )
    .withColumn(
        "longitude_eda",
        F.expr("""
            try_cast(
                regexp_extract(location_1, ',\\\\s*(-?[0-9.]+)\\\\)', 1)
                AS DOUBLE
            )
        """)
    )
    .withColumn(
        "day_of_week",
        F.date_format("recorded_date", "EEEE")
    )
)

numeric_stats = eda.select(
    "consideration_amount_numeric",
    "latitude_eda",
    "longitude_eda"
).summary(
    "count", "mean", "stddev", "min", "25%", "50%", "75%", "max"
)

display(numeric_stats)

doc_type_stats = (
    eda.groupBy("doc_type")
    .agg(F.count("*").alias("total_filings"))
    .withColumn(
        "percentage",
        F.round(
            F.col("total_filings") / total_rows * 100, 2
        )
    )
    .orderBy(F.desc("total_filings"))
)

display(doc_type_stats)

weekday_stats = (
    eda.groupBy("day_of_week")
    .agg(F.count("*").alias("total_filings"))
    .orderBy(F.desc("total_filings"))
)

display(weekday_stats)

monthly_eda = (
    eda.groupBy(
        "year",
        "recorded_month",
        "month_start"
    )
    .agg(
        F.count("*").alias("total_filings"),
        F.countDistinct("pin_clean").alias("unique_pins"),
        F.sum(
            F.col("has_location").cast("int")
        ).alias("geocoded_filings")
    )
    .orderBy("month_start")
)

display(monthly_eda)

In [0]:

# ==========================================
# SECTION 9: OUTLIER DETECTION USING IQR
# ==========================================

amount_data = eda.filter(
    F.col("consideration_amount_numeric").isNotNull()
)

quartiles = amount_data.approxQuantile(
    "consideration_amount_numeric",
    [0.25, 0.50, 0.75],
    0.01
)

if len(quartiles) == 3:

    q1, median, q3 = quartiles

    iqr = q3 - q1

    lower_bound = q1 - (1.5 * iqr)
    upper_bound = q3 + (1.5 * iqr)

    outliers = amount_data.filter(
        (F.col("consideration_amount_numeric") < lower_bound) |
        (F.col("consideration_amount_numeric") > upper_bound)
    )

    outlier_count = outliers.count()

    print("IQR OUTLIER ANALYSIS")
    print("Q1:", q1)
    print("Median:", median)
    print("Q3:", q3)
    print("IQR:", iqr)
    print("Lower Bound:", lower_bound)
    print("Upper Bound:", upper_bound)
    print("Potential Outliers:", outlier_count)

    display(
        outliers.select(
            "pin_clean",
            "doc_number",
            "consideration_amount_numeric"
        )
    )

else:
    q1 = median = q3 = iqr = lower_bound = upper_bound = None
    outlier_count = 0

    print("Insufficient numeric values for IQR analysis.")

negative_amounts = amount_data.filter(
    F.col("consideration_amount_numeric") < 0
)

print("Negative consideration amounts:", negative_amounts.count())


In [0]:

# ==========================================
# SECTION 10: PREPARE VISUALIZATION DATA
# ==========================================

import matplotlib.pyplot as plt
import os

output_dir = "/Volumes/cs483eviction/raw/cook_county_foreclosures/eda_outputs"

os.makedirs(output_dir, exist_ok=True)

monthly_pd = monthly_eda.toPandas()
doc_type_pd = doc_type_stats.toPandas()

missing_pd = (
    fc.select([
        F.count(
            F.when(F.col(c).isNull(), 1)
        ).alias(c)
        for c in [
            "Pin",
            "doc_number",
            "doc_type",
            "recorded_date",
            "execution_date",
            "consideration_amount",
            "location_1"
        ]
    ])
    .toPandas()
    .melt(
        var_name="Column",
        value_name="Missing Records"
    )
)

amount_pd = (
    eda.select("consideration_amount_numeric")
    .where(F.col("consideration_amount_numeric").isNotNull())
    .toPandas()
)

locations_pd = (
    eda.filter(
        F.col("latitude_eda").isNotNull() &
        F.col("longitude_eda").isNotNull()
    )
    .select("latitude_eda", "longitude_eda")
    .sample(False, 0.35, seed=42)
    .limit(15000)
    .toPandas()
)

print("Visualization data prepared.")
print("Monthly observations:", len(monthly_pd))
print("Document categories:", len(doc_type_pd))
print("Consideration amount observations:", len(amount_pd))
print("Coordinate observations plotted:", len(locations_pd))

# Monthly foreclosure fillings graph

plt.figure(figsize=(11, 6))

plt.bar(
    monthly_pd["recorded_month"],
    monthly_pd["total_filings"]
)

plt.title("Monthly Foreclosure Filings in Cook County (2012)")
plt.xlabel("Month")
plt.ylabel("Number of Recorded Filings")
plt.xticks(range(1, 13))
plt.grid(axis="y", alpha=0.3)

plt.tight_layout()

plt.savefig(
    f"{output_dir}/monthly_foreclosure_filings.png",
    dpi=300
)

plt.show()

# Consideration amount distribution graph

plt.figure(figsize=(10, 6))

plt.hist(
    amount_pd["consideration_amount_numeric"],
    bins=15,
    edgecolor="black"
)

plt.title("Distribution of Available Consideration Amounts")
plt.xlabel("Consideration Amount")
plt.ylabel("Frequency")
plt.grid(axis="y", alpha=0.3)

plt.tight_layout()

plt.savefig(
    f"{output_dir}/consideration_amount_distribution.png",
    dpi=300
)

plt.show()

# Missing values by column graph


plt.figure(figsize=(11, 6))

plt.bar(
    missing_pd["Column"],
    missing_pd["Missing Records"]
)

plt.title("Missing Values by Dataset Column")
plt.xlabel("Dataset Column")
plt.ylabel("Number of Missing Records")
plt.xticks(rotation=35, ha="right")
plt.grid(axis="y", alpha=0.3)

plt.tight_layout()

plt.savefig(
    f"{output_dir}/missing_values.png",
    dpi=300
)

plt.show()

# Geographic distribution of foreclosure fillings

plt.figure(figsize=(9, 8))

plt.scatter(
    locations_pd["longitude_eda"],
    locations_pd["latitude_eda"],
    alpha=0.3,
    s=8
)

plt.title("Geographic Distribution of Foreclosure Filings")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.grid(alpha=0.3)

plt.tight_layout()

plt.savefig(
    f"{output_dir}/foreclosure_geographic_distribution.png",
    dpi=300
)

plt.show()


In [0]:

# ==========================================
# SECTION 11: FEATURE SCALING DIAGNOSTICS
# ==========================================

scaling_stats = eda.select(
    "consideration_amount_numeric",
    "latitude_eda",
    "longitude_eda"
).summary(
    "count",
    "mean",
    "stddev",
    "min",
    "max"
)

print("FEATURE SCALING DIAGNOSTICS")
display(scaling_stats)


In [0]:

# ============================================================
# SECTION 12: AUTOMATICALLY GENERATE EDA MARKDOWN REPORT
# ============================================================

%pip install tabulate

import os
import pandas as pd
from datetime import datetime

# Prepare output directory

output_dir = "/Volumes/cs483eviction/raw/cook_county_foreclosures/eda_outputs"
os.makedirs(output_dir, exist_ok=True)

# Collect dataset overview

total_rows = fc.count()
total_columns = len(fc.columns)

date_values = fc.agg(
    F.min("recorded_date").alias("earliest"),
    F.max("recorded_date").alias("latest")
).first()

earliest_date = date_values["earliest"]
latest_date = date_values["latest"]

# Count records with valid coordinates
coordinate_count = eda.filter(
    F.col("latitude_eda").isNotNull() &
    F.col("longitude_eda").isNotNull()
).count()

coordinate_percentage = (
    coordinate_count / total_rows * 100 if total_rows > 0 else 0
)

# Convert summary DataFrames to Pandas

numeric_stats_pd = numeric_stats.toPandas()
scaling_stats_pd = scaling_stats.toPandas()
doc_type_pd = doc_type_stats.toPandas()
monthly_pd = monthly_eda.toPandas()
weekday_pd = weekday_stats.toPandas()

# Explicitly convert column names to strings
for pdf in [
    numeric_stats_pd,
    scaling_stats_pd,
    doc_type_pd,
    monthly_pd,
    weekday_pd
]:
    pdf.columns = pdf.columns.astype(str)

# Convert tables into Markdown

numeric_stats_text = numeric_stats_pd.to_markdown(index=False)
scaling_stats_text = scaling_stats_pd.to_markdown(index=False)
doc_type_text = doc_type_pd.to_markdown(index=False)
monthly_text = monthly_pd.to_markdown(index=False)
weekday_text = weekday_pd.to_markdown(index=False)

# Prepare missing-value statistics

missing_rows = []

for column in original_columns:
    missing_count = fc.filter(
        F.col(column).isNull() |
        (F.trim(F.col(column).cast("string")) == "")
    ).count()

    missing_percentage = (
        missing_count / total_rows * 100
        if total_rows > 0 else 0
    )

    missing_rows.append({
        "Column": column,
        "Missing Values": missing_count,
        "Missing Percentage": round(missing_percentage, 2)
    })

missing_pd = pd.DataFrame(missing_rows)
missing_text = missing_pd.to_markdown(index=False)

# Prepare duplicate statistics

exact_duplicates = (
    total_rows - fc.dropDuplicates().count()
)

repeated_doc_numbers = (
    total_rows - fc.select("doc_number").distinct().count()
)

repeated_pins = (
    total_rows - fc.select("pin_clean").distinct().count()
)

repeated_pin_doc_pairs = (
    total_rows -
    fc.select("pin_clean", "doc_number").distinct().count()
)

# Prepare document type findings

most_common_type = (
    doc_type_pd.iloc[0]["doc_type"]
    if not doc_type_pd.empty else "Not available"
)

most_common_type_count = (
    doc_type_pd.iloc[0]["total_filings"]
    if not doc_type_pd.empty else 0
)

# Prepare busiest month

if not monthly_pd.empty:
    busiest_month = monthly_pd.loc[
        monthly_pd["total_filings"].idxmax()
    ]

    busiest_month_name = pd.to_datetime(
        busiest_month["month_start"]
    ).strftime("%B %Y")

    busiest_month_count = int(
        busiest_month["total_filings"]
    )
else:
    busiest_month_name = "Not available"
    busiest_month_count = 0

# Prepare outlier statistics

outlier_text = "Outlier analysis was not available."

if "outlier_count" in globals():
    if q1 is not None:
        outlier_text = f"""
- First quartile (Q1): {q1:,.2f}
- Median: {median:,.2f}
- Third quartile (Q3): {q3:,.2f}
- Interquartile range (IQR): {iqr:,.2f}
- Lower outlier boundary: {lower_bound:,.2f}
- Upper outlier boundary: {upper_bound:,.2f}
- Potential outliers identified: {outlier_count:,}
"""
    else:
        outlier_text = "Insufficient numeric data for IQR analysis."

# Prepare date validation statistics

records_outside_2012 = fc.filter(
    F.year("recorded_date") != 2012
).count()

execution_after_recorded = fc.filter(
    F.col("execution_date") > F.col("recorded_date")
).count()

execution_missing_count = fc.filter(
    F.col("execution_date").isNull()
).count()

# Build the Markdown report

report = f"""
# Cook County Foreclosure Dataset
## Exploratory Data Analysis (EDA) Report

### 1. Dataset Overview

This project analyzes the Cook County Recorder of Deeds
foreclosure dataset covering January 1, 2012 through
November 29, 2012.

The dataset contains foreclosure-related legal filings,
primarily lis pendens foreclosure notices. These records
indicate pending foreclosure proceedings and should not
automatically be interpreted as completed foreclosures
or repossessions.

| Dataset Metric | Result |
|---|---:|
| Total records | {total_rows:,} |
| Total columns in cleaned DataFrame | {total_columns} |
| Earliest recorded date | {earliest_date} |
| Latest recorded date | {latest_date} |
| Records with valid coordinates | {coordinate_count:,} |
| Geographic coverage percentage | {coordinate_percentage:.2f}% |

### 2. Data Cleaning and Preparation

The following cleaning and transformation steps were
performed using Apache Spark:

- Renamed columns for easier processing.
- Converted recorded dates into timestamp format.
- Converted execution dates into timestamps where valid.
- Standardized PIN values by removing hyphens and padding
  identifiers.
- Extracted year and month from recorded dates.
- Extracted latitude and longitude from location information
  for geographic analysis.
- Created indicators for missing execution dates,
  consideration amounts, and geographic coordinates.
- Converted consideration amounts into numeric values for
  descriptive analysis, treating malformed or blank values
  as NULL.

No records were intentionally removed during the cleaning
process.

### 3. Missing-Value Analysis

The following table summarizes missing or blank values
in the original dataset fields.

{missing_text}

Consideration Amount should be interpreted cautiously
because a large proportion of its values are missing.
Geographic analysis is limited to records with usable
location coordinates.

### 4. Duplicate Analysis

| Duplicate Check | Result |
|---|---:|
| Exact duplicate records | {exact_duplicates:,} |
| Repeated document number occurrences beyond first | {repeated_doc_numbers:,} |
| Repeated PIN occurrences beyond first | {repeated_pins:,} |
| Repeated PIN-document pair occurrences beyond first | {repeated_pin_doc_pairs:,} |

Repeated PINs or document numbers do not necessarily
represent duplicate records. A property may be associated
with multiple filings, and amended or corrected filings
may legitimately reference existing documents.

### 5. Descriptive Statistics

The following table summarizes the numeric variables
available for analysis.

{numeric_stats_text}

The consideration amount statistics are based only on
values that could be converted into numeric data.
Latitude and longitude statistics are based on records
with extractable coordinates.

### 6. Foreclosure Document Type Distribution

{doc_type_text}

The most frequently occurring document type was
**{most_common_type}**, with {most_common_type_count:,}
records.

### 7. Monthly Foreclosure Filing Analysis

{monthly_text}

The month with the highest number of recorded filings was
**{busiest_month_name}**, with {busiest_month_count:,}
filings.

### 8. Day-of-Week Analysis

{weekday_text}

This table shows how recorded filings are distributed
across the days of the week.

### 9. Outlier Analysis

{outlier_text}

The IQR method was used to identify potential outliers
in consideration amounts. Potential outliers were flagged
for investigation and were not automatically removed.

### 10. Date Validation

| Validation Check | Result |
|---|---:|
| Recorded dates outside 2012 | {records_outside_2012:,} |
| Execution dates after recorded dates | {execution_after_recorded:,} |
| Missing or invalid execution dates after cleaning | {execution_missing_count:,} |

Execution dates that could not be parsed or fell outside
the accepted date range in the cleaning step were treated
as missing. Therefore, the missing count includes both
originally missing and filtered execution dates.

### 11. Visualizations

The following four graphs were generated using Matplotlib:

#### Monthly Foreclosure Filings

![Monthly Foreclosure Filings](monthly_foreclosure_filings.png)

#### Consideration Amount Distribution

![Consideration Amount Distribution](consideration_amount_distribution.png)

#### Missing Values

![Missing Values](missing_values.png)

#### Geographic Distribution

![Geographic Distribution](foreclosure_geographic_distribution.png)

### 12. Feature Scaling Assessment

{scaling_stats_text}

The numeric variables have different measurement units
and ranges. Feature scaling may be appropriate for
distance-based or gradient-based machine learning
algorithms. Tree-based algorithms generally do not
require standardization.

No feature scaling was applied as part of this
descriptive analysis.

### 13. Data Quality and Limitations

Several limitations should be considered:

- Consideration Amount is heavily incomplete.
- Some records do not contain usable geographic
  coordinates.
- Repeated PINs and document numbers are not necessarily
  errors.
- Foreclosure notices do not establish that a property
  was ultimately foreclosed or repossessed.
- The dataset covers a specific period in 2012 and
  should not be interpreted as representing current
  foreclosure activity.

### 14. Apache Spark Implementation

Apache Spark was used to perform data loading,
transformations, data-quality checks, aggregations,
and descriptive statistics in the Databricks environment.

Screenshots of the Spark execution DAGs and job stages
should be added here to demonstrate distributed
processing.

### 15. Conclusion

This exploratory analysis provides an overview of
Cook County foreclosure-related filings recorded during
2012. The analysis examined document types, monthly
filing activity, missing values, repeated identifiers,
numeric variables, and geographic coverage.

The findings also highlight data-quality limitations
that should be considered before conducting additional
analysis or building predictive models.
"""

# Save report

report_path = os.path.join(output_dir, "EDA.md")

with open(report_path, "w", encoding="utf-8") as file:
    file.write(report)

print("EDA report successfully generated!")
print("Report location:", report_path)

print("\nGenerated graph files:")

for filename in [
    "monthly_foreclosure_filings.png",
    "foreclosure_document_types.png",
    "consideration_amount_distribution.png",
    "missing_values.png",
    "foreclosure_geographic_distribution.png"
]:
    graph_path = os.path.join(output_dir, filename)

    if os.path.exists(graph_path):
        print("FOUND:", filename)
    else:
        print("MISSING:", filename)

print("\nDataset summary:")
print("Total records:", total_rows)
print("Total columns:", total_columns)
print("Geocoded records:", coordinate_count)
print("Exact duplicates:", exact_duplicates)

print("\nReport saved to:", report_path)
